<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> 책의 보조 코드 by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# GPT-4를 사용한 리플렉션 튜닝을 통한 지시 데이터 개선

- 이 노트북은 OpenAI의 GPT-4 API를 사용하여 [Reflection-Tuning: Data Recycling Improves LLM Instruction-Tuning](https://arxiv.org/abs/2310.11716) 논문의 데이터셋 개선 과정을 구현합니다

![](https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/reflection-tuning/reflection-tuning.webp)

- 원본 논문에서 연구자들은 [Alpaca](https://huggingface.co/datasets/tatsu-lab/alpaca)와 [WizardLM](https://huggingface.co/datasets/WizardLMTeam/WizardLM_evol_instruct_70k) 지시 미세조정 데이터셋을 개선했습니다. 이 노트북에서는 [7장에서 사용된 지시 데이터셋](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch07/01_main-chapter-code/instruction-data.json)을 개선합니다 (그러나 Alpaca와 같은 형식이므로 동일한 코드가 Alpaca 데이터셋에서도 작동합니다)

- 예상되는 데이터셋 형식은 다음과 같습니다:

```python
    {
        "instruction": "Edit the following sentence for grammar.",
        "input": "He go to the park every day.",
        "output": "He goes to the park every day."
    },
    {
        "instruction": "Convert 45 kilometers to meters.",
        "input": "",
        "output": "45 kilometers is 45000 meters."
    },
```

> 이 노트북은 저자들이 GPT API를 사용하여 기존 데이터셋을 개선한 논문의 접근법을 재현한다는 점에 주목하세요. 그러나 [OpenAI 이용약관](https://openai.com/policies/row-terms-of-use/)에 명시된 대로 GPT API로 생성된 데이터는 OpenAI와 경쟁하는 모델을 개발하는 데 사용할 수 없다는 점을 인지하는 것이 중요합니다: "할 수 없는 것... 출력을 사용하여 OpenAI와 경쟁하는 모델을 개발하는 것." 관련 논의는 [여기](https://www.reddit.com/r/LocalLLaMA/comments/17vbg1f/does_openai_tos_prohibit_generating_datasets_for/)에서 찾을 수 있습니다.

In [ ]:
# pip install -r requirements-extra.txt

In [1]:
from importlib.metadata import version

pkgs = [
    "openai",  # OpenAI API
    "tqdm",    # 진행률 표시줄
]

for p in pkgs:
    print(f"{p} version: {version(p)}")

openai version: 1.30.3
tqdm version: 4.66.4


## OpenAI API 테스트

- 먼저 OpenAI API가 올바르게 설정되어 있는지 테스트해보겠습니다
- 아직 계정이 없다면 https://platform.openai.com/ 에서 계정을 만들어야 합니다
- GPT-4 API는 무료가 아니므로 계정에 일부 자금을 충전해야 합니다 (https://platform.openai.com/settings/organization/billing/overview 참조)
- 이 노트북에 정확히 표시된 대로 코드를 실행하면 이 글을 쓰는 시점에서 GPT-4o-mini를 사용하여 약 $0.03 (3센트)가 소요됩니다
- 위의 두 방법론을 7장 지시 데이터셋의 모든 1100개 항목에 적용하면 약 $0.60 (60센트)가 소요됩니다

- 먼저 OpenAI API 비밀 키를 제공해야 합니다. 이는 https://platform.openai.com/api-keys 에서 찾을 수 있습니다
- 이 키를 다른 사람과 공유하지 않도록 주의하세요
- 이 비밀 키(`"sk-..."`)를 이 폴더의 `config.json` 파일에 추가하세요

In [2]:
import json
from openai import OpenAI

# JSON 파일에서 API 키를 로드합니다.
# "sk-..."를 https://platform.openai.com/api-keys의 실제 API 키로 바꾸세요
with open("config.json", "r") as config_file:
    config = json.load(config_file)
    api_key = config["OPENAI_API_KEY"]

client = OpenAI(api_key=api_key)

- 먼저 간단한 예제로 API를 시도해서 의도한 대로 작동하는지 확인해보겠습니다:

In [3]:
def run_chatgpt(prompt, client, model="gpt-4o-mini", system_prompt=None):
    # system_prompt가 제공되면 시스템 메시지 정의
    messages = []
    
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    
    # 메시지에 사용자 프롬프트 추가
    messages.append({"role": "user", "content": prompt})

    # API 호출
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.0,
        seed=123,
    )
    
    # 모델의 응답 반환
    return response.choices[0].message.content


prompt = "Respond with 'hello world' if you got this message."
run_chatgpt(prompt, client)

'hello world'

## JSON 항목 로드

- 다음으로 지시 데이터셋을 로드하고 처리해보겠습니다
- 여기서는 테스트 데이터셋과 모델 응답을 다음과 같이 로드할 수 있는 JSON 파일로 저장했다고 가정합니다:

In [4]:
from pathlib import Path


json_file = Path("..") / "01_main-chapter-code" / "instruction-data.json"

with open(json_file, "r") as file:
    json_data = json.load(file)

print("Number of entries:", len(json_data))

Number of entries: 1100


- 데이터셋 항목 중 하나를 출력해서 구조를 확인해보겠습니다:

In [5]:
from pprint import pp as pprint

pprint(json_data[0])

{'instruction': 'Evaluate the following phrase by transforming it into the '
                'spelling given.',
 'input': 'freind --> friend',
 'output': 'The spelling of the given phrase "freind" is incorrect, the '
           'correct spelling is "friend".'}


## 지시 개선

- Reflection-Tuning 저자들은 두 가지 접근법을 공유했습니다: (1) 지시 개선과 (2) 응답 개선
- 주어진 데이터셋에서 지시를 개선하는 것부터 시작해보겠습니다
- 아래는 이 데이터셋 개선을 위해 GPT-4 모델에 대한 입력을 형식화하는 [Reflection-Tuning 저장소](https://github.com/tianyi-lab/Reflection_Tuning/blob/main/reflection_code/reflect_response.py)의 작은 유틸리티 함수입니다

In [6]:
def instr_prompt_no_input(ins, outp):

    sys_prompt = "You are a helpful, precise but picky assistant for checking the quality of a given instruction."
    prompt_template = "[Instruction]\n{ins}\n\n[The Start of Answer]\n{outp}\n\n[The End of Answer]\n\n[System]\n{criteria}\n\n"
    criteria = "We would like you to answer several questions related to the quality of a given instruction. \n" + \
                "1. Why this instruction is not good? First analyse the instruction based on Complexity of the Topic, Level of Detail Required, Knowledge Required, Ambiguity of the Instruction and Logical Reasoning or Problem-Solving Involved. \n" + \
                "Then analyse why this answer is not good for the given instruction? Analyse based on the Helpfulness, Relevance, Accuracy and Level of Details. \n" + \
                "Finally analyse why this bad instruction lead to a bad answer. " +\
                "2. Based on the reason you provided, generate a new and complete instruction which is complex and difficult to answer directly. " + \
                "Make sure the new instruction is relevent but independent to the original instruction, which can be answered without knowing the original instruction, put the new instruction in the format of [New Instruction] your instruction [End]" +\
                "3. Answer the newly generated instruction as detailed as possible, in the format of [New Answer] your answer [End] \n"
    prompt = prompt_template.format(
        ins=ins, outp=outp, criteria=criteria
    )
    return sys_prompt, prompt

- 어떻게 작동하는지 보기 위해 데이터셋 항목 `json_data[2]`를 고려해보겠습니다

In [7]:
pprint(json_data[2])

{'instruction': 'Convert 45 kilometers to meters.',
 'input': '',
 'output': '45 kilometers is 45000 meters.'}


- 위에서 정의한 `instr_prompt_no_input` 함수를 사용하여 지시를 다음과 같이 개선할 수 있습니다:

In [8]:
entry = json_data[2]

system_prompt, prompt = instr_prompt_no_input(ins=entry["instruction"], outp=entry["output"])
output = run_chatgpt(prompt=prompt, client=client, system_prompt=system_prompt)

print(output)

1. **Analysis of the Instruction:**

   - **Complexity of the Topic:** The topic of converting kilometers to meters is relatively simple and straightforward, as it involves basic unit conversion.
   - **Level of Detail Required:** The instruction does not require much detail; it simply asks for a conversion without any additional context or explanation.
   - **Knowledge Required:** Basic knowledge of metric units and their conversions is required, which is common knowledge.
   - **Ambiguity of the Instruction:** The instruction is clear and unambiguous; it specifies exactly what needs to be converted.
   - **Logical Reasoning or Problem-Solving Involved:** There is minimal logical reasoning involved, as the conversion factor (1 kilometer = 1000 meters) is a standard fact.

   **Analysis of the Answer:**

   - **Helpfulness:** The answer is helpful in that it provides the correct conversion.
   - **Relevance:** The answer is relevant to the instruction, as it directly addresses the conv

- 응답이 매우 장황한데, 이는 분석 목적에 유용합니다. 또한 연쇄 사고 프롬프팅 접근법을 통해 GPT-4 모델이 개선사항을 만드는 데 도움이 됩니다
- 그러나 개선된 데이터셋을 구성하기 위해서는 실제로 분석이 아닌 새로운 지시와 출력에만 관심이 있습니다
- 모델의 개선된 지시와 출력을 추출하기 위해 [Reflection-Tuning 저장소](https://github.com/tianyi-lab/Reflection_Tuning/blob/main/reflection_code/reflect_response.py)의 다음 유틸리티 코드를 사용할 수 있습니다

In [9]:
import re

def extract_ins(text, no_input=True):
    if '[New Instruction]' in text:
        pattern = r'(\[New Instruction\])(.*?)(\[End\]|\[New Answer\]|New Answer:)'
    else:
        pattern = r'(New Instruction:)(.*?)(\[End\]|\[New Answer\]|New Answer:)'
    segments = re.findall(pattern, text, re.DOTALL)
    if len(segments) == 0:
        seg_ins = ''
    else:
        seg_ins = segments[0][1].strip()
    if seg_ins.endswith("\n\n3."):
        seg_ins = seg_ins[:-4]
    return seg_ins


def extract_oup(text, no_input=True):
    if '[New Answer]' in text:
        pattern = r'(\[New Answer\])(.*?)(\[End\]|$)'
    else:
        pattern = r'(New Answer:)(.*?)(\[End\]|$)'
        # pattern = r'(\[New Answer\]|New Answer:)(.*?)(\[End\]|$)'
    segments = re.findall(pattern, text, re.DOTALL)
    if len(segments) == 0:
        seg_oup = ''
    else:
        seg_oup = segments[0][1].strip()
    return seg_oup


def extract_instruction(text):
    if text == '':
        return []
    seg_ins = extract_ins(text, no_input=True)
    seg_oup = extract_oup(text, no_input=True)
    return [seg_ins, seg_oup]

- 이 유틸리티 함수들을 사용하여 앞서 생성된 긴 GPT-4 출력에서 개선된 지시와 응답을 추출해보겠습니다:

In [10]:
new_instr, new_outp = extract_instruction(output)

In [11]:
print(new_instr)

Explain the significance of the metric system in global trade and provide examples of how unit conversions can impact international business transactions.


In [12]:
print(new_outp)

The metric system, also known as the International System of Units (SI), is a decimal-based system of measurement that is used globally. Its significance in global trade lies in its standardization, which facilitates international communication and commerce. 

   One of the primary advantages of the metric system is that it is universally recognized, which reduces confusion and errors in measurement. For example, when a company in the United States imports goods from Europe, the specifications for those goods are often provided in metric units. If the U.S. company is accustomed to using imperial units (like inches or pounds), they must convert these measurements to ensure compatibility. 

   Unit conversions can significantly impact international business transactions. For instance, if a manufacturer orders 100 kilograms of a product but mistakenly interprets it as 100 pounds, they will receive a much smaller quantity than intended, leading to production delays and financial losses. 



- 지시 개선은 현재 `"input"` 필드가 없는 데이터셋 항목에 대해서만 구현되어 있습니다

## 응답 개선

- 비슷한 방식으로 Reflection-Tuning 개선 과정을 데이터셋 응답(즉, "output" 필드)에 특별히 적용할 수도 있습니다
- 아래는 데이터셋 개선을 위해 GPT-4 모델에 대한 입력을 형식화하는 [Reflection-Tuning 저장소](https://github.com/tianyi-lab/Reflection_Tuning/blob/main/reflection_code/reflect_response.py)의 두 개의 작은 유틸리티 함수입니다

In [13]:
def res_gen_prompt_no_input(ins, outp):

    sys_prompt = "You are a helpful, precise but picky assistant for checking the quality of the answer to a given instruction."
    prompt_template = "[Instruction]\n{ins}\n\n[The Start of Answer]\n{outp}\n\n[The End of Answer]\n\n[System]\n{criteria}\n\n"
    criteria = "We would like you to answer several questions related to the quality of the answer to the given instruction. \n" + \
                "1. Why this answer is not good for the given instruction? Analyse based on the Helpfulness, Relevance, Accuracy and Level of Details. \n" + \
                "2. Based on the reason you provided, generate a better answer, new and complete, as detailed as possible, in the format of [Better Answer] your answer [End] \n" 
    prompt = prompt_template.format(
        ins=ins, outp=outp, criteria=criteria
    )
    return sys_prompt, prompt


def res_gen_prompt_input(ins, inp, outp):

    sys_prompt = "You are a helpful and precise assistant for checking the quality of the answer to a given instruction and its input."
    prompt_template = "[Instruction]\n{ins}\n\n[The Start of Input]\n{inp}\n\n[The End of Input]\n\n[The Start of Answer]\n{outp}\n\n[The End of Answer]\n\n[System]\n{criteria}\n\n"
    criteria = "We would like you to answer several questions related to the quality of the answer to the given instruction and corresponding input. \n" + \
                "1. Why this answer is not good for the given instruction and corresponding input? Analyse based on the Helpfulness, Relevance, Accuracy and Level of Details. \n" + \
                "2. Based on the reason you provided, generate a better answer, new and complete, as detailed as possible, in the format of [Better Answer] your answer [End] \n" 
    prompt = prompt_template.format(
        ins=ins, inp=inp, outp=outp, criteria=criteria
    )
    return sys_prompt, prompt

- 다시 데이터셋 항목 중 하나에 적용해서 어떻게 작동하는지 확인하고 개선된 응답을 생성해보겠습니다:

In [14]:
entry = json_data[2]

system_prompt, prompt = res_gen_prompt_no_input(ins=entry["instruction"], outp=entry["output"])
output = run_chatgpt(prompt=prompt, client=client, system_prompt=system_prompt)

print(output)

1. The answer provided is not good for the given instruction for several reasons:

- **Helpfulness**: While the answer does provide the correct conversion, it lacks any explanation or context. A more helpful answer would include a brief explanation of the conversion process, which would aid understanding.

- **Relevance**: The answer is relevant in that it addresses the instruction to convert kilometers to meters, but it could be more relevant by including the conversion factor used (1 kilometer = 1000 meters).

- **Accuracy**: The answer is accurate in terms of the numerical conversion (45 kilometers = 45000 meters). However, it could be misleading if the reader does not understand how the conversion was derived.

- **Level of Details**: The answer is very brief and lacks detail. A more detailed response would include the conversion factor and a step-by-step explanation of how the conversion is performed.

2. [Better Answer] To convert kilometers to meters, you can use the conversion 

- 위에서 볼 수 있듯이 응답에는 원본 응답의 분석이 포함되어 있습니다. [Reflection-Tuning 저장소](https://github.com/tianyi-lab/Reflection_Tuning/blob/main/reflection_code/reflect_response.py)의 다음 유틸리티 함수를 사용하여 새 응답을 추출할 수 있습니다

In [15]:
def extract_response(text):
    if text.count('[Better Answer]') >= 2:
        pattern = r'\[(Better Answer)\](.*?)(\[End\]|\[Better Answer\]|$)'
        segments = re.findall(pattern, text, re.DOTALL)
    else:
        # pattern = r'\[(Better Answer)\](.*?)\[End\]'
        pattern = r'\[(Better Answer)\](.*?)(\[End\]|End|$)'
        segments = re.findall(pattern, text, re.DOTALL)
    return [segment[1].strip() for segment in segments]

In [16]:
response = extract_response(output)[0]
print(response)

To convert kilometers to meters, you can use the conversion factor that 1 kilometer is equal to 1000 meters. Therefore, to convert 45 kilometers to meters, you multiply 45 by 1000. 

So, 45 kilometers × 1000 meters/kilometer = 45000 meters. 

Thus, 45 kilometers is equal to 45000 meters.


## 데이터셋 개선

- 이제 실제 데이터셋에 지시 리플렉션과 응답 리플렉션 기법을 적용해보겠습니다
- 참고: 여기서는 데모 목적으로 작은 데이터 하위 집합에만 적용합니다. 전체 데이터셋에 적용하려면

```python
data_to_process = json_data[:3]
```

을

```python
data_to_process = json_data
```

로 변경하세요

### 지시 리플렉션

- 다음 코드는 원본 데이터셋의 지시에 대한 데이터셋 개선을 위한 Reflection-Tuning 방법론을 적용합니다

In [17]:
data_to_process = json_data[:3]

In [18]:
from tqdm import tqdm


def reflect_instructions(json_data, client):
    new_json_data = [] 
    
    for entry in tqdm(json_data):
        
        if not entry["input"]:
            system_prompt, prompt = instr_prompt_no_input(ins=entry["instruction"], outp=entry["output"])
            output = run_chatgpt(prompt=prompt, client=client, system_prompt=system_prompt)
            new_instr, new_outp = extract_instruction(output)
            new_entry = {"instruction": new_instr, "input": "", "output": new_outp}
            new_json_data.append(new_entry)
        else:
            new_json_data.append(entry)

    return new_json_data

In [19]:
data_to_process = json_data[:3]

new_json_data = reflect_instructions(data_to_process, client)

100%|█████████████████████████████████████████████| 3/3 [00:06<00:00,  2.17s/it]


In [20]:
for i in new_json_data[:3]:
    pprint(i)
    print("\n\n")

{'instruction': 'Evaluate the following phrase by transforming it into the '
                'spelling given.',
 'input': 'freind --> friend',
 'output': 'The spelling of the given phrase "freind" is incorrect, the '
           'correct spelling is "friend".'}



{'instruction': 'Edit the following sentence for grammar.',
 'input': 'He go to the park every day.',
 'output': 'He goes to the park every day.'}



{'instruction': 'Explain the significance of understanding metric conversions '
                'in scientific research, and provide an example of how a '
                'miscalculation in unit conversion could impact experimental '
                'results.',
 'input': '',
 'output': 'Understanding metric conversions is crucial in scientific research '
           'because accurate measurements are fundamental to the validity of '
           'experimental results. The metric system is widely used in '
           'scientific disciplines due to its ease of use and universal '
    

- 새 데이터셋을 저장해보겠습니다:

In [21]:
with open("instruction-reflected.json", "w") as file:
    json.dump(new_json_data, file, indent=4)

### 응답 리플렉션

- 이제 응답 리플렉션에 대해서도 같은 작업을 해보겠습니다:

In [22]:
data_to_process = json_data[:3]

In [23]:
def reflect_responses(json_data, client):
    new_json_data = [] 
    
    for entry in tqdm(json_data):
        
        if not entry["input"]:
            system_prompt, prompt = res_gen_prompt_no_input(ins=entry["instruction"], outp=entry["output"])
            output = run_chatgpt(prompt=prompt, client=client, system_prompt=system_prompt)
            new_response = extract_response(output)

            if not len(new_response):
                new_response = entry["output"]
                      
            new_entry = {"instruction": entry["instruction"], "input": "", "output": new_response[0]}
            new_json_data.append(new_entry)

        else:
            system_prompt, prompt = res_gen_prompt_input(ins=entry["instruction"], inp=entry["input"], outp=entry["output"])
            output = run_chatgpt(prompt=prompt, client=client, system_prompt=system_prompt)
            new_response = extract_response(output)

            if not len(new_response):
                new_response = entry["output"]

            new_entry = {"instruction": entry["instruction"], "input": entry["input"], "output": new_response[0]}
            new_json_data.append(new_entry)

    return new_json_data

In [24]:
new_json_data = reflect_responses(data_to_process, client)

100%|█████████████████████████████████████████████| 3/3 [00:07<00:00,  2.40s/it]


In [25]:
for i in new_json_data[:3]:
    pprint(i)
    print("\n\n")

{'instruction': 'Evaluate the following phrase by transforming it into the '
                'spelling given.',
 'input': 'freind --> friend',
 'output': 'The input phrase "freind" contains a spelling error. The correct '
           'transformation of the word is as follows: "freind" should be '
           'corrected to "friend." Therefore, the correct spelling is '
           '"friend."'}



{'instruction': 'Edit the following sentence for grammar.',
 'input': 'He go to the park every day.',
 'output': 'The original sentence "He go to the park every day" contains a '
           'grammatical error in the verb form. The correct form should be "He '
           'goes to the park every day." This is because the subject "He" is '
           'third person singular, and in English, the verb "to go" changes to '
           '"goes" when used with third person singular subjects. Therefore, '
           'the corrected sentence is grammatically accurate and maintains the '
           'original mea

- 새 데이터셋을 저장해보겠습니다:

In [26]:
with open("response-reflected.json", "w") as file:
    json.dump(new_json_data, file, indent=4)

## 개선된 지시 데이터 생성

- 위의 두 방법론을 7장 지시 데이터셋의 모든 1100개 항목에 적용하면 약 $0.60 (60센트)가 소요됩니다
- GitHub 저장소가 데이터셋 파일로 부풀려지는 것을 방지하기 위해 결과 데이터셋 파일은 Google Drive에서 사용할 수 있습니다:
  - [instruction-reflected.json](https://drive.google.com/file/d/1c1QnuTdt9nP1u51vBn4_b05mWR_ZNGBv/view?usp=sharing)
  - [response-reflected.json](https://drive.google.com/file/d/1RNckTZ2ELcdUoJtaylao6NvyZPMtNv1v/view?usp=sharing)